# Milestone 3 - Varying Number of Clusters

Sweeps `k = 5, 10, 20, 50` using existing embeddings and distributed GMM from Milestone 2 blueprint.

Measured metrics: runtime, iteration/convergence, communication overhead, WSS, silhouette, purity, ARI, weighted Jaccard, clinical coherence, memory, and I/O.

## AWS EC2 Tips

- Keep rank count fixed when comparing different k.
- For k=50, use more memory if needed.
- If runtime is high, start with `tfidf_lsa50` or reduce row count, then scale up.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

for _var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_var, "1")

def find_project_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "Final Datasets").exists() and (candidate / "Milestone 3").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
M3_DIR = PROJECT_ROOT / "Milestone 3"
RUNNER = M3_DIR / "m3_experiment_runner.py"
RESULTS_DIR = M3_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)

print(PROJECT_ROOT)

In [ ]:
EMBEDDING = "bge"
DISTRIBUTION = "block"
MPI_RANKS = 4
FIXED_ROWS = 8000
K_VALUES = [5, 10, 20, 50]
N_INIT = 1
MAX_ITER = 120

OUT_CSV = RESULTS_DIR / "m3_cluster_sweep_metrics.csv"
if OUT_CSV.exists():
    OUT_CSV.unlink()

print(f"Output: {OUT_CSV}")

In [ ]:
def run_mpi_experiment(k):
    cmd = [
        "mpirun", "-np", str(MPI_RANKS),
        "python", str(RUNNER),
        "--mode", "k",
        "--embedding", EMBEDDING,
        "--distribution", DISTRIBUTION,
        "--n-clusters", str(k),
        "--rows", str(FIXED_ROWS),
        "--n-init", str(N_INIT),
        "--max-iter", str(MAX_ITER),
        "--tag", f"k{k}",
        "--out-file", str(OUT_CSV),
    ]
    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

for k in K_VALUES:
    run_mpi_experiment(k)

print("Cluster sweep runs complete.")

In [ ]:
df = pd.read_csv(OUT_CSV).sort_values("n_clusters").reset_index(drop=True)
df["comm_overhead_pct"] = 100.0 * df["communication_fraction"]
df["convergence_rate_iter_per_s"] = df["gmm_iterations"] / df["gmm_total_seconds"]

display_cols = [
    "n_clusters", "rows_used", "mpi_ranks",
    "gmm_total_seconds", "avg_iteration_seconds", "gmm_iterations", "convergence_rate_iter_per_s",
    "communication_seconds", "comm_overhead_pct",
    "wss", "silhouette_known", "purity", "ari", "jaccard_weighted",
    "coherence_top_share_mean", "coherence_entropy_mean",
    "peak_rss_mb_max_rank", "io_load_seconds", "io_throughput_mb_s"
]
print(df[display_cols].to_string(index=False))

df.to_csv(OUT_CSV, index=False)
print(f"Updated metrics saved to {OUT_CSV}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].plot(df["n_clusters"], df["gmm_total_seconds"], marker="o")
axes[0, 0].set_title("Runtime vs k")
axes[0, 0].set_xlabel("k")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df["n_clusters"], df["wss"], marker="o")
axes[0, 1].set_title("WSS vs k")
axes[0, 1].set_xlabel("k")
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(df["n_clusters"], df["silhouette_known"], marker="o", label="Silhouette")
axes[0, 2].plot(df["n_clusters"], df["purity"], marker="s", label="Purity")
axes[0, 2].set_title("Quality vs k")
axes[0, 2].set_xlabel("k")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].plot(df["n_clusters"], df["ari"], marker="o", label="ARI")
axes[1, 0].plot(df["n_clusters"], df["jaccard_weighted"], marker="s", label="Jaccard")
axes[1, 0].set_title("Agreement vs k")
axes[1, 0].set_xlabel("k")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df["n_clusters"], df["comm_overhead_pct"], marker="o", label="Comm overhead %")
axes[1, 1].plot(df["n_clusters"], df["avg_iteration_seconds"], marker="s", label="Avg iter sec")
axes[1, 1].set_title("Comm and Iteration vs k")
axes[1, 1].set_xlabel("k")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(df["n_clusters"], df["coherence_top_share_mean"], marker="o", label="Top-share")
axes[1, 2].plot(df["n_clusters"], df["coherence_entropy_mean"], marker="s", label="Entropy")
axes[1, 2].set_title("Clinical Coherence vs k")
axes[1, 2].set_xlabel("k")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / "m3_cluster_sweep_plots.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(plot_path)

# Milestone 3 - Varying Number of Clusters

This notebook evaluates clustering behavior for varying cluster counts (`k = 5, 10, 20, 50`) using existing embeddings and the Milestone 2 distributed GMM blueprint.

Measured metrics:
- Runtime and convergence (iterations, convergence rate)
- Communication overhead
- Cluster quality: WSS, silhouette, purity, ARI, weighted Jaccard
- Clinical/domain coherence: dominant cohort share, entropy-based coherence
- Memory and I/O characteristics

## EC2 Practical Guidance

- For `k=50`, memory and communication load increase; prefer at least 16 GB RAM if using larger row counts.
- Keep rank count fixed in this notebook when comparing different `k` values.
- If runtime grows too much, start with `tfidf_lsa50` or lower row count, then scale up.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for _var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_var, "1")

plt.rcParams["figure.dpi"] = 120

def find_project_root(start=None):
    p = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "Final Datasets").exists() and (candidate / "Milestone 3").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
M3_DIR = PROJECT_ROOT / "Milestone 3"
RUNNER = M3_DIR / "m3_experiment_runner.py"
RESULTS_DIR = M3_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

In [ ]:
# Experiment configuration
EMBEDDING = "bge"
DISTRIBUTION = "block"
MPI_RANKS = 4
FIXED_ROWS = 8000
K_VALUES = [5, 10, 20, 50]
N_INIT = 1
MAX_ITER = 120

OUT_CSV = RESULTS_DIR / "m3_cluster_sweep_metrics.csv"
if OUT_CSV.exists():
    OUT_CSV.unlink()

print(f"Output: {OUT_CSV}")

In [ ]:
def run_mpi_experiment(k):
    cmd = [
        "mpirun", "-np", str(MPI_RANKS),
        "python", str(RUNNER),
        "--mode", "k",
        "--embedding", EMBEDDING,
        "--distribution", DISTRIBUTION,
        "--n-clusters", str(k),
        "--rows", str(FIXED_ROWS),
        "--n-init", str(N_INIT),
        "--max-iter", str(MAX_ITER),
        "--tag", f"k{k}",
        "--out-file", str(OUT_CSV),
    ]
    print("Running:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

for k in K_VALUES:
    run_mpi_experiment(k)

print("Cluster sweep runs complete.")

In [ ]:
df = pd.read_csv(OUT_CSV).sort_values("n_clusters").reset_index(drop=True)
df["comm_overhead_pct"] = 100.0 * df["communication_fraction"]
df["convergence_rate_iter_per_s"] = df["gmm_iterations"] / df["gmm_total_seconds"]

display_cols = [
    "n_clusters", "rows_used", "mpi_ranks",
    "gmm_total_seconds", "avg_iteration_seconds", "gmm_iterations", "convergence_rate_iter_per_s",
    "communication_seconds", "comm_overhead_pct",
    "wss", "silhouette_known", "purity", "ari", "jaccard_weighted",
    "coherence_top_share_mean", "coherence_entropy_mean",
    "peak_rss_mb_max_rank", "io_load_seconds", "io_throughput_mb_s"
]
display(df[display_cols])

df.to_csv(OUT_CSV, index=False)
print(f"Updated metrics saved to {OUT_CSV}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].plot(df["n_clusters"], df["gmm_total_seconds"], marker="o")
axes[0, 0].set_title("Runtime vs k")
axes[0, 0].set_xlabel("k")
axes[0, 0].set_ylabel("seconds")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df["n_clusters"], df["wss"], marker="o")
axes[0, 1].set_title("WSS vs k")
axes[0, 1].set_xlabel("k")
axes[0, 1].set_ylabel("WSS")
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(df["n_clusters"], df["silhouette_known"], marker="o", label="Silhouette")
axes[0, 2].plot(df["n_clusters"], df["purity"], marker="s", label="Purity")
axes[0, 2].set_title("Quality Metrics vs k")
axes[0, 2].set_xlabel("k")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].plot(df["n_clusters"], df["ari"], marker="o", label="ARI")
axes[1, 0].plot(df["n_clusters"], df["jaccard_weighted"], marker="s", label="Jaccard")
axes[1, 0].set_title("Agreement Metrics vs k")
axes[1, 0].set_xlabel("k")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df["n_clusters"], df["comm_overhead_pct"], marker="o", label="Comm overhead %")
axes[1, 1].plot(df["n_clusters"], df["avg_iteration_seconds"], marker="s", label="Avg iter sec")
axes[1, 1].set_title("Communication/Iteration vs k")
axes[1, 1].set_xlabel("k")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(df["n_clusters"], df["coherence_top_share_mean"], marker="o", label="Top-share coherence")
axes[1, 2].plot(df["n_clusters"], df["coherence_entropy_mean"], marker="s", label="Entropy coherence")
axes[1, 2].set_title("Clinical Coherence vs k")
axes[1, 2].set_xlabel("k")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / "m3_cluster_sweep_plots.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(plot_path)

## Notes

- Choose the best `k` by balancing quality (`silhouette`, `purity`, `ARI`) and scalability costs (`runtime`, `comm_overhead_pct`).
- For reporting, include both numerical table and the generated plots.